In [1]:
from ase.io import read
from ase.units import Ha
from ase_dirac import DIRAC

# Input definition and calculator constructor methods
In the previous (part of the) Tutorial, we defined the calculation setup using the internal dictionary representation of the `DIRAC` calculator object. In this tutorial we will show a small addition to abstract that layer when diract input files are already availiable. 
##  Direct construction
The first "layer" is the construction of an ASE `Atoms` object with the calculator defined, ready to interact with the functionalities of ASE. For this we define two files `molecule.xyz` and `dirac_example.inp`:

In [2]:
xyz_string = """2
F2
O      0 0 0
O      0.00000       -0.75545     -0.47116
"""

fscc_str = """ **DIRAC
    .TITLE
    Molecular Te. DEA from the dianion.
    .WAVE F
    **HAMILTONIAN
    .LVCORR
    **WAVE FUNCTIONS
    .SCF
    .RELCCSD
    ! because we want this
    *SCF
    .PRINT 
    1
    !.CLOSED SHELL
    !54 52
    **RELCCSD
    .ENERGY
    
    
    .PRINT
    1
    .FOCKSP
    *CCFSPC
    .DOIE2
    .NACTH
    1 1 0 0
    .MAXIT
    200
    **MOLECULE
    *BASIS
    .DEFAULT
    dyall.3zp
    ! **Moltra
    ! .Active
    ! energy -10.0 20.0 1.0
    *END OF
"""

with open("molecule.xyz", "w") as f:
    f.write(xyz_string)

with open("dirac_example.inp", "w") as f:
    f.write(fscc_str)

We can now define an atoms object from an input file and a coordinate, and run the calculations as in the previous (part of the) tutorial:

In [ ]:
molecule = DIRAC.ASE_atoms_from_inpfile_and_coords("dirac_example.inp", "molecule.xyz")
molecule.get_potential_energy() / Ha

{'dirac': {'.TITLE': 'Molecular Te. DEA from the dianion.', '.WAVE F': '', '.4index': ''}, 'hamiltonian': {'.LVCORR': ''}, 'wave_function': {'.SCF': '', '.RELCCSD': '', '*SCF': {'.PRINT': '1'}, '**RELCCSD': {'.ENERGY': '', '.PRINT': '1', '.FOCKSP': ''}, '*CCFSPC': {'.DOIE2': '', '.NACTH': '1 1 0 0', '.MAXIT': '200'}}, 'molecule': {'*BASIS': {'.DEFAULT': 'dyall.3zp'}, '*END OF': {}, '*charge': {'.charge': '0'}, '*symmetry': {'.nosym': '#'}}, 'integrals': {'.nucmod': '1'}, 'moltra': {'.active': 'all'}}


Making possible to run whole DIRAC calculations with only two lines of code. Also, the previous molecule object can be obtained with a predefined `Atoms` object if necesssary (for example if some previous interaction via ASE is required): 

In [ ]:
# molecule_2 = read("molecule.xyz", format="xyz")
# molecule_2 = DIRAC.ASE_atoms_from_inpfile("dirac_example.inp", molecule_2)
# molecule_2.get_potential_energy() / Ha

 starting DIRAC for molecule dirac_example.xyz with input dirac_example.inp
  creating archive file  dirac_example.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal


-148.89408341013115


---

## Calculator constructors
The previous definition is very simple, but under the hood the following takes place. ASE's `Atoms` requires a `calc` object to run calculations. In our case we use the custom `DIRAC` calculator object. Therefore, it is necessary to initialize this object and attach it to the `Atoms`. Once the file (or string) are defined, it is possible to construct the object as:

```python
ase_molecule = read("molecule.xyz", format="xyz")
ase_molecule.calc = DIRAC.from_file("dirac_example.inp")
```

Or :

```python
ase_molecule.calc = DIRAC.from_string(fscc_str)
```


In [6]:
ase_molecule = read("molecule.xyz", format="xyz")
ase_molecule.calc = DIRAC.calculator_from_file("dirac_example.inp")

# Also it can be called from string as:
# ase_molecule.calc = DIRAC.calculator_from_string(fscc_str)

Where we can see the internal dictionary definition that this calculator presents:

In [7]:
param_list = [
    f"{key}, {ase_molecule.calc.parameters[key]}\n"
    for key in ase_molecule.calc.parameters.keys()
]

print("".join(param_list))

dirac, {'.TITLE': 'Molecular Te. DEA from the dianion.', '.WAVE F': '', '.4INDEX': '', '.4index': ''}
hamiltonian, {'.LVCORR': ''}
wave_function, {'.SCF': '', '.RELCCSD': '', '*SCF': {'.PRINT': '1'}, '**RELCCSD': {'.ENERGY': '', '.PRINT': '1', '.FOCKSP': ''}, '*CCFSPC': {'.DOIE2': '', '.NACTH': '1 1 0 0', '.MAXIT': '200'}}
molecule, {'*BASIS': {'.DEFAULT': 'dyall.3zp'}, '*END OF': {}, '*CHARGE': {'.CHARGE': '0'}, '*SYMMETRY': {'.NOSYM': '#'}, '*charge': {'.charge': '0'}, '*symmetry': {'.nosym': '#'}}
integrals, {'.NUCMOD': '1'}
moltra, {'.ACTIVE': 'all', '*END OF INPUT': {}}



If for whatever reason it is necessary to modify the input parameters, it is possible to start from a template `.inp` file and then modify the dictionary keyword-value pairs to override the defaults:

In [12]:
ase_molecule.calc = DIRAC.calculator_from_file("dirac_example.inp")

ase_molecule.calc.parameters.moltra.update({".active": "energy -10 20 1"})
ase_molecule.calc.parameters.molecule.update({"*symmetry": {".LINEAR": ""}})

param_list = [
    f"{key}, {ase_molecule.calc.parameters[key]}\n"
    for key in ase_molecule.calc.parameters.keys()
]

print("".join(param_list))

try:
    ase_molecule.calc.calculate()
except TypeError:
    ase_molecule.get_potential_energy() / Ha

dirac, {'.TITLE': 'Molecular Te. DEA from the dianion.', '.WAVE F': '', '.4INDEX': '', '.4index': ''}
hamiltonian, {'.LVCORR': ''}
wave_function, {'.SCF': '', '.RELCCSD': '', '*SCF': {'.PRINT': '1'}, '**RELCCSD': {'.ENERGY': '', '.PRINT': '1', '.FOCKSP': ''}, '*CCFSPC': {'.DOIE2': '', '.NACTH': '1 1 0 0', '.MAXIT': '200'}}
molecule, {'*BASIS': {'.DEFAULT': 'dyall.3zp'}, '*END OF': {}, '*CHARGE': {'.CHARGE': '0'}, '*SYMMETRY': {'.NOSYM': '#'}, '*charge': {'.charge': '0'}, '*symmetry': {'.LINEAR': ''}}
integrals, {'.NUCMOD': '1'}
moltra, {'.ACTIVE': 'all', '*END OF INPUT': {}, '.active': 'energy -10 20 1'}

 starting DIRAC for molecule dirac.xyz with input dirac.inp
  creating archive file  dirac.tgz
  archived working files: ['MOLECULE.XYZ', 'DIRAC.INP', 'AOPROPER', 'AOMOMAT', 'dirac.xml']
  exit           : normal



---
# Calculation results and I/O
Once a calculation has finished, it is possible to interact with the results directly using the `DIRAC` calculator object. We can check some common data such as energies and eigenvalues direcly by calling:


In [13]:
ase_molecule.calc.results

{'SCF_energy': -1393.5473329223928,
 'MP2_energy': -148.27627346665363,
 'CCSD_energy': -148.89408341013115,
 'energy': -4051.6143804174135,
 'scf_energy': array([-149.18871037]),
 'scf_eigenvalues': array([-1.57030684e+05, -7.74102831e+04, -5.20712002e+04, -4.29013088e+04,
        -3.95804520e+04, -3.83798113e+04, -3.83628648e+04, -3.82549734e+04,
        -3.82549160e+04, -3.78951841e+04, -3.77872538e+04, -3.77502114e+04,
        -3.77500875e+04, -3.77048580e+04, -3.76417582e+04, -3.76247202e+04,
        -3.76244764e+04, -3.76231048e+04, -3.75913594e+04, -3.75866056e+04,
        -3.75831163e+04, -3.75830649e+04, -3.75741035e+04, -3.75732271e+04,
        -3.75713682e+04, -3.75701398e+04, -3.75697566e+04, -3.75691523e+04,
        -3.75684166e+04, -3.75673873e+04, -3.75670542e+04, -3.75633875e+04,
        -3.75623364e+04, -3.75617728e+04, -3.75617599e+04, -3.75615962e+04,
        -3.75612220e+04, -3.75609953e+04, -3.75606878e+04, -3.75599602e+04,
        -3.75595401e+04, -3.75591090e+04,

## h5 functionality from the calculator 
Under the hood what is happening is that the `read_results()` method in the calculator was extended to retrieve some data from the `CHECKPOINT.h5` file generated in the calculation: 

```python
def read_results(self): 
    """Read energy from DIRAC output file. Try to extract from h5 if availiable"""
    out_file = self.label + ".out"
    output = read_dirac_out(out_file)
    self.results = output
    self._extract_common_results_from_h5()
```

Where `_extract_common_results_from_h5()` extracts from the associated `.h5` file what now is an arbitrary list of (probably) values of interest: 

```python
        commonly_requested_results: dict = {
            "result/wavefunctions/scf/energy": "scf_energy",
            "result/wavefunctions/scf/mobasis/eigenvalues": 'scf_eigenvalues',
        }
```

A property `DIRAC.h5_contents` property was added to list all availiable properties in the `.h5` file: 

In [10]:
ase_molecule.calc.h5_contents[0:10]

['Group: input',
 'Group: input/aobasis',
 'Group: input/aobasis/1',
 'Dataset: input/aobasis/1/angular (1,)',
 'Dataset: input/aobasis/1/aobasis_id (1,)',
 'Dataset: input/aobasis/1/center (72,)',
 'Dataset: input/aobasis/1/contractions (24,)',
 'Dataset: input/aobasis/1/exponents (24,)',
 'Dataset: input/aobasis/1/n_ao (1,)',
 'Dataset: input/aobasis/1/n_cont (24,)']

And a `.extract_h5_value` method was added in order to extract properties by key:

In [11]:
n_ao = ase_molecule.calc.extract_h5_value("result/operators/ao_matrices/BETAMAT FFFT")
print(repr(n_ao))

array([0.        , 0.        , 0.        , ..., 0.2301652 , 0.4878467 ,
       1.33938936], shape=(60726,))


For more information on how to manipulate DIRAC's `.h5` files, chech the `.h5` tutorials.